# Laboratorio F: MNIST con PyTorch (sin CNN)

**Redes Neuronales II - Especializacion en Inteligencia Artificial**
**Autor:** Harold Steven Rueda Lara
**Objetivo:** Resolver el mismo problema de reconocimiento de digitos MNIST del Laboratorio E, ahora con PyTorch, donde el ciclo de entrenamiento (forward, loss, backward, actualizacion) se escribe de forma explicita en vez de estar oculto dentro de un `fit()`.

> **Importante:** este notebook debe ejecutarse en Google Colab (Entorno de ejecucion > Ejecutar todas) para descargar el dataset MNIST completo y generar las evidencias de ejecucion que van en el documento tecnico.


## Paso 33. Importar y cargar MNIST

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

## Paso 34. Crear DataLoaders
Un *batch* es un pequeno grupo de ejemplos; aqui la red procesa 64 imagenes por lote.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Paso 35. Definir la red (arquitectura 784 -> 128 -> 10, igual que en Keras)

In [ ]:
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.capa1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.salida = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.capa1(x)
        x = self.relu(x)
        x = self.salida(x)
        return x

modelo_pytorch = RedMNIST()
print(modelo_pytorch)

## Paso 36. Crear perdida y optimizador
**Importante:** este bloque debe ejecutarse antes del entrenamiento; si no, aparece `NameError: optimizador is not defined`.

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_pytorch.parameters(), lr=0.001)
print("Modelo, criterio y optimizador listos")

## Paso 37. Entrenar
`error.backward()` calcula los gradientes (backpropagation); `optimizador.step()` usa esos gradientes para modificar los pesos.

In [ ]:
epochs = 3
for epoch in range(epochs):
    modelo_pytorch.train()
    error_total = 0
    for imagenes, etiquetas in train_loader:
        # 1. Limpiar gradientes anteriores
        optimizador.zero_grad()
        # 2. Forward
        salidas = modelo_pytorch(imagenes)
        # 3. Loss
        error = criterio(salidas, etiquetas)
        # 4. Backpropagation
        error.backward()
        # 5. Actualizar pesos
        optimizador.step()
        error_total += error.item()

    promedio_error = error_total / len(train_loader)
    print("Epoca:", epoch + 1, "- Error:", round(promedio_error, 4))

## Paso 38. Evaluar

In [ ]:
modelo_pytorch.eval()
correctas = 0
total = 0
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = modelo_pytorch(imagenes)
        _, predicciones = torch.max(salidas, 1)
        total += etiquetas.size(0)
        correctas += (predicciones == etiquetas).sum().item()

accuracy_pytorch = correctas / total
print("Accuracy en prueba:", accuracy_pytorch)

## Paso 39. Ver una prediccion individual

In [ ]:
import matplotlib.pyplot as plt

imagen, etiqueta_real = test_dataset[0]
entrada = imagen.unsqueeze(0)
modelo_pytorch.eval()
with torch.no_grad():
    salida = modelo_pytorch(entrada)
prediccion = torch.argmax(salida, dim=1).item()

print("Numero real:", etiqueta_real)
print("Numero predicho:", prediccion)

plt.imshow(imagen.squeeze(), cmap="gray")
plt.title(f"Real: {etiqueta_real} | Prediccion: {prediccion}")
plt.axis("off")
plt.show()

## Conclusion del laboratorio
PyTorch resuelve el mismo problema que Keras en el Laboratorio E, pero exponiendo explicitamente el ciclo de entrenamiento (`zero_grad -> forward -> loss -> backward -> step`), lo que hace mas visible donde ocurre cada parte del proceso: forward propagation, calculo de la perdida, backpropagation y actualizacion de pesos.

## Comparacion de enfoques (NumPy manual vs Keras vs PyTorch)

| Accion | NumPy manual | TensorFlow/Keras | PyTorch |
|---|---|---|---|
| Crear capas | Matrices `W` y `b` | `Dense` | `nn.Linear` |
| ReLU | `np.maximum` | `activation="relu"` | `nn.ReLU` |
| Sigmoide | Funcion propia | `activation="sigmoid"` | `nn.Sigmoid` si se requiere |
| Loss | Funcion propia | `compile(loss=...)` | `CrossEntropyLoss` |
| Backpropagation | Derivadas manuales | Automatico en `fit()` | `error.backward()` |
| Actualizar pesos | `W -= lr*dW` | Optimizer automatico | `optimizer.step()` |
| Entrenar | `for` manual | `fit()` | `for` explicito |
| Evaluar | Calculo manual | `evaluate()` | `eval()` + `no_grad()` |

No se trata de decidir cual framework es "mejor": ambos automatizan los mismos fundamentos matematicos que se implementaron a mano en los Laboratorios A y B.